In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from sklearn.metrics.pairwise import cosine_similarity
import math


In [ ]:


df = pd.read_json("/content/output.json")
print(df.columns)

Index(['job_id', 'job_title', 'career_category', 'company_name', 'job_level',
       'location_city', 'location_district', 'salary_min_usd',
       'salary_max_usd', 'salary_text', 'experience_required', 'education',
       'employment_type', 'openings', 'tech_stack', 'job_summary',
       'main_responsibilities', 'requirements', 'preferred_qualifications',
       'benefits', 'work_location_detail', 'working_time', 'probation_period',
       'application_deadline', 'post_date', 'source_site', 'is_synthetic',
       'full_job_post_text'],
      dtype='object')


In [ ]:
df["location_city"].value_counts()

,count
location_city,
Hồ Chí Minh,41
Đà Nẵng,35
Hà Nội,21


In [ ]:

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   job_id                    97 non-null     int64 
 1   job_title                 97 non-null     object
 2   career_category           97 non-null     object
 3   company_name              97 non-null     object
 4   job_level                 97 non-null     object
 5   location_city             97 non-null     object
 6   location_district         97 non-null     object
 7   salary_min_usd            97 non-null     int64 
 8   salary_max_usd            97 non-null     int64 
 9   salary_text               97 non-null     object
 10  experience_required       97 non-null     object
 11  education                 97 non-null     object
 12  employment_type           97 non-null     object
 13  openings                  97 non-null     int64 
 14  tech_stack                97

In [ ]:
df.describe()

,job_id,salary_min_usd,salary_max_usd,openings
count,97.000000,97.000000,97.000000,97.000000
mean,49.000000,1506.185567,2900.000000,2.783505
std,28.145456,548.827241,1121.754578,1.408507
min,1.000000,800.000000,1500.000000,1.000000
25%,25.000000,1000.000000,1800.000000,2.000000
50%,49.000000,1400.000000,2700.000000,3.000000
75%,73.000000,2100.000000,4200.000000,4.000000
max,97.000000,2300.000000,4500.000000,5.000000


In [ ]:
df.isna().sum()

,0
job_id,0
job_title,0
career_category,0
company_name,0
job_level,0
location_city,0
location_district,0
salary_min_usd,0
salary_max_usd,0
salary_text,0


In [ ]:
cols = [
    "main_responsibilities",
    "requirements",
    "preferred_qualifications"
]

print(df[cols].head(3).to_string())

                                                                                                                                                                                                                                               main_responsibilities                                                                                                                                                                                                                                                            requirements                                                                                                                                                                         preferred_qualifications
0    - Thiết kế, tối ưu và bảo trì hệ thống ở mức module hoặc service\n- Xử lý bug production và cải thiện chất lượng hệ thống\n- Viết code sạch, dễ bảo trì và thực hiện code review định kỳ\n- Phát triển tính năng mới theo yêu cầu sản phẩm và tài liệu kỹ thuật  - Nắm chắc OOP, cấu 

In [ ]:


def extract_exp_range(text):
    if pd.isna(text):
        return (0, 0)

    text = str(text).lower()

    # case: 2-4 năm
    match_range = re.findall(r"\d+", text)
    if len(match_range) >= 2:
        return (int(match_range[0]), int(match_range[1]))

    # case: 4+ năm
    if "+" in text and match_range:
        return (int(match_range[0]), int(match_range[0]) + 5)  # estimate

    # case: 1 năm
    if match_range:
        return (int(match_range[0]), int(match_range[0]))

    # case: fresher / no exp
    if "fresher" in text or "không" in text:
        return (0, 0)

    return (0, 0)

df[["exp_min", "exp_max"]] = df["experience_required"].apply(
    lambda x: pd.Series(extract_exp_range(x))
)



In [ ]:
df.head(5)

,job_id,job_title,career_category,company_name,job_level,location_city,location_district,salary_min_usd,salary_max_usd,salary_text,...,work_location_detail,working_time,probation_period,application_deadline,post_date,source_site,is_synthetic,full_job_post_text,exp_min,exp_max
0,1,Backend Developer,Software Engineering,NSTAGE,Junior,Đà Nẵng,Sơn Trà,900,1700,900-1700 USD,...,"44 Hai Bà Trưng, Sơn Trà, Đà Nẵng","Thứ 2 - Thứ 6 (08:30 - 17:30), nghỉ trưa 1 tiếng",Thỏa thuận theo năng lực,2026-04-13,2026-03-21,TopCV-like synthetic dataset,True,Mô tả công việc\nCông ty đang tìm kiếm Backend...,1,2
1,2,Product Owner,Product,Zalo,Junior,Đà Nẵng,Sơn Trà,1100,1500,1100-1500 USD,...,"27 Hai Bà Trưng, Sơn Trà, Đà Nẵng","Thứ 2 - Thứ 6 (08:00 - 17:00), có thể hybrid 2...",85-100% lương thử việc trong 2 tháng,2026-04-07,2026-02-26,TopCV-like synthetic dataset,True,Mô tả công việc\nChúng tôi cần Product Owner c...,1,2
2,3,Fullstack Developer,Software Engineering,Amanotes,Junior,Đà Nẵng,Hải Châu,1100,2000,1100-2000 USD,...,"32 Hai Bà Trưng, Hải Châu, Đà Nẵng","Thứ 2 - Thứ 6 (08:00 - 17:00), có thể hybrid 2...",Thỏa thuận theo năng lực,2026-04-05,2026-03-09,TopCV-like synthetic dataset,True,Mô tả công việc\nCông ty đang tìm kiếm Fullsta...,1,2
3,4,Frontend Developer,Software Engineering,Sky Mavis,Junior,Hồ Chí Minh,Quận 1,800,1700,800-1700 USD,...,"59 Nguyễn Trãi, Quận 1, Hồ Chí Minh","Thứ 2 - Thứ 6 (08:00 - 17:00), có thể hybrid 2...",2 tháng,2026-04-04,2026-02-27,TopCV-like synthetic dataset,True,Mô tả công việc\nVị trí Frontend Developer sẽ ...,1,2
4,5,Backend Developer,Software Engineering,NAB Innovation Centre,Senior,Đà Nẵng,Ngũ Hành Sơn,2100,4200,2100-4200 USD,...,"71 Láng Hạ, Ngũ Hành Sơn, Đà Nẵng","Thứ 2 - Thứ 6 (09:00 - 18:00), giờ vào linh ho...",85-100% lương thử việc trong 2 tháng,2026-04-24,2026-03-11,TopCV-like synthetic dataset,True,Mô tả công việc\nVị trí Backend Developer sẽ đ...,4,9


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

match_cols = [
    "job_title",
    "career_category",
    "tech_stack",
    "job_summary",
    "main_responsibilities",
    "requirements",
    "education"
]

for col in match_cols:
    df[col] = df[col].apply(clean_text)

In [ ]:
df["job_text_match"] = (
    df["job_title"] + " " +
    df["job_title"] + " " +
    df["tech_stack"] + " " +
    df["tech_stack"] + " " +
    df["career_category"] + " " +
    df["job_summary"] + " " +
    df["main_responsibilities"] + " " +
    df["requirements"] + " " +
    df["education"]
)

df["job_text_match"] = df["job_text_match"].str.replace(r"\s+", " ", regex=True).str.strip()
df["job_text_match"] = df["job_text_match"].str.replace(r"[-•–]", " ", regex=True).str.strip()

In [ ]:
print(df[[ "job_text_match"]].head(1).to_string())

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       job_text_match
0  backend developer backend developer java, kafka, sql, spring boot, redis java, kafka, sql, spring boot, redis software engineering công ty đang tìm kiếm backen

# Load Model

In [ ]:


def precision_at_k(pred_ids, true_ids, k=5):
    pred_topk = pred_ids[:k]
    hit = sum(1 for x in pred_topk if x in true_ids)
    return hit / k

def hitrate_at_k(pred_ids, true_ids, k=5):
    pred_topk = pred_ids[:k]
    return 1.0 if any(x in true_ids for x in pred_topk) else 0.0

def mrr(pred_ids, true_ids):
    for rank, job_id in enumerate(pred_ids, start=1):
        if job_id in true_ids:
            return 1.0 / rank
    return 0.0

In [ ]:
test_cases = [
    {
        "cv_text": "backend developer 2 năm kinh nghiệm java spring boot kafka redis sql xây dựng api debug production code review",
        "true_job_ids": [1, 5, 37, 41,22]
    },
    {
        "cv_text": "data engineer 2 năm kinh nghiệm spark python sql airflow etl pipeline data warehouse kafka",
        "true_job_ids": [6, 14, 29, 56]
    },
    {
        "cv_text": "business analyst có kinh nghiệm viết tài liệu nghiệp vụ uml sql agile làm việc với stakeholder và uat",
        "true_job_ids": [12, 18, 43]
    }
]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model2 = SentenceTransformer("BAAI/bge-m3")

job_texts = ["passage: " + x for x in df["job_text_match"].tolist()]
job_embeddings2 = model2.encode(
    job_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
results = []

for case in test_cases:
    cv_text = clean_text(case["cv_text"])

    cv_embedding = model2.encode(
    ["query: " + cv_text],
    normalize_embeddings=True)
    scores = cosine_similarity(cv_embedding, job_embeddings2)[0]
    temp = df.copy()
    temp["score"] = scores
    ranked = temp.sort_values("score", ascending=False)

    pred_ids = ranked["job_id"].tolist()
    true_ids = case["true_job_ids"]

    results.append({
        "cv_text": case["cv_text"][:60] + "...",
        "P@5": precision_at_k(pred_ids, true_ids, k=5),
        "HitRate@5": hitrate_at_k(pred_ids, true_ids, k=5),
        "MRR": mrr(pred_ids, true_ids)
    })

eval_df = pd.DataFrame(results)
print(eval_df)
print("\nAverage metrics:")
print(eval_df[["P@5", "HitRate@5", "MRR"]].mean())

                                             cv_text  P@5  HitRate@5  MRR
0  backend developer 2 năm kinh nghiệm java sprin...  0.8        1.0  0.5
1  data engineer 2 năm kinh nghiệm spark python s...  0.6        1.0  1.0
2  business analyst có kinh nghiệm viết tài liệu ...  0.6        1.0  1.0

Average metrics:
P@5          0.666667
HitRate@5    1.000000
MRR          0.833333
dtype: float64


In [ ]:


def experience_penalty_asymmetric(candidate_years, exp_min, exp_max,
                                  under_penalty=0.05,
                                  over_penalty=0.015,
                                  growth="quadratic",
                                  max_penalty=0.25):
    if pd.isna(exp_min):
        exp_min = 0
    if pd.isna(exp_max):
        exp_max = 999

    exp_min = float(exp_min)
    exp_max = float(exp_max)
    candidate_years = float(candidate_years)

    if exp_min > exp_max:
        exp_min, exp_max = exp_max, exp_min

    if exp_min <= candidate_years <= exp_max:
        return 0.0

    if candidate_years < exp_min:
        distance = exp_min - candidate_years
        base = under_penalty
    else:
        distance = candidate_years - exp_max
        base = over_penalty

    if growth == "linear":
        penalty = base * distance
    elif growth == "quadratic":
        penalty = base * (distance ** 2)
    elif growth == "log":
        penalty = base * math.log1p(distance)
    else:
        raise ValueError("growth must be one of: linear, quadratic, log")

    penalty = min(penalty, max_penalty)
    return -penalty

In [ ]:


def evaluate_job_recommender(
    test_cases,
    df_jobs,
    model,
    job_embeddings,
    years_experience=2,
    penalty_func=None,
    query_prefix="query: ",
    top_k=5,
    penalty_returns_negative=True,
    show_case_scores=False
):


    results = []

    for case in test_cases:
        raw_cv_text = case["cv_text"]
        true_ids = case["true_job_ids"]

        cv_text = clean_text(raw_cv_text)

        if query_prefix:
            model_input = [query_prefix + cv_text]
        else:
            model_input = [cv_text]

        cv_embedding = model.encode(
            model_input,
            normalize_embeddings=True
        )

        raw_scores = cosine_similarity(cv_embedding, job_embeddings)[0]

        temp = df_jobs.copy()
        temp["raw_score"] = raw_scores

        if penalty_func is not None:
            temp["exp_penalty"] = temp.apply(
                lambda r: penalty_func(
                    years_experience,
                    r["exp_min"],
                    r["exp_max"]
                ),
                axis=1
            )
        else:
            temp["exp_penalty"] = 0.0

        if penalty_returns_negative:
            temp["final_score"] = temp["raw_score"] + temp["exp_penalty"]
        else:
            temp["final_score"] = temp["raw_score"] - temp["exp_penalty"]

        ranked = temp.sort_values("final_score", ascending=False)
        pred_ids = ranked["job_id"].tolist()

        if show_case_scores:
            print("=" * 100)
            print("CV:", raw_cv_text[:120], "...")
            cols_to_show = [
                c for c in [
                    "job_id", "job_title", "experience_required",
                    "raw_score", "exp_penalty", "final_score"
                ] if c in ranked.columns
            ]
            print(ranked[cols_to_show].head(top_k))

        results.append({
            "cv_text": raw_cv_text[:60] + "...",
            "years_experience": years_experience,
            f"P@{top_k}": precision_at_k(pred_ids, true_ids, k=top_k),
            f"HitRate@{top_k}": hitrate_at_k(pred_ids, true_ids, k=top_k),
            "MRR": mrr(pred_ids, true_ids)
        })

    eval_df = pd.DataFrame(results)
    avg_metrics = eval_df[[f"P@{top_k}", f"HitRate@{top_k}", "MRR"]].mean()

    return eval_df, avg_metrics

In [ ]:
def location_penalty_soft(candidate_city=None,
                          job_city=None,
                          diff_city_penalty=0.03):

    def norm_text(x):
        if pd.isna(x) or x is None:
            return None
        return str(x).strip().lower()

    candidate_city = norm_text(candidate_city)
    job_city = norm_text(job_city)

    # không truyền preference địa điểm thì không phạt
    if candidate_city is None:
        return 0.0

    # nếu khác city -> phạt nhẹ
    if candidate_city is not None and candidate_city != job_city:
        return -diff_city_penalty

    # chỉ cần city giống là không phạt
    return 0.0

In [ ]:
def recommend_jobs(cv_text, years_experience, top_n=10,
                   df_jobs=None,
                   model=None,
                   job_embeddings=None,
                   under_penalty=0.05,
                   over_penalty=0.015,
                   growth="quadratic",
                   max_penalty=0.25,
                   candidate_city=None,
                   diff_city_penalty=0.04):


    if df_jobs is None:
        raise ValueError("df_jobs không được None")
    if model is None:
        raise ValueError("model không được None")
    if job_embeddings is None:
        raise ValueError("job_embeddings không được None")

    cleaned_cv = clean_text(cv_text)

    cv_embedding = model.encode(
        ["query: " + cleaned_cv],
        normalize_embeddings=True
    )

    raw_scores = (cv_embedding @ job_embeddings.T)[0]

    temp = df_jobs.copy()
    temp["raw_score"] = raw_scores

    # penalty kinh nghiệm
    temp["exp_penalty"] = temp.apply(
        lambda r: experience_penalty_asymmetric(
            candidate_years=years_experience,
            exp_min=r["exp_min"],
            exp_max=r["exp_max"],
            under_penalty=under_penalty,
            over_penalty=over_penalty,
            growth=growth,
            max_penalty=max_penalty
        ),
        axis=1
    )

    # penalty địa điểm
    temp["location_penalty"] = temp.apply(
        lambda r: location_penalty_soft(
            candidate_city=candidate_city,
            job_city=r.get("location_city"),
            diff_city_penalty=diff_city_penalty
        ),
        axis=1
    )

    temp["final_score"] = (
        temp["raw_score"]
        + temp["exp_penalty"]
        + temp["location_penalty"]
    )

    ranked = temp.sort_values("final_score", ascending=False).head(top_n)

    cols = [
        "job_id",
        "job_title",
        "company_name",
        "location_city",
        "location_district",
        "experience_required",
        "exp_min",
        "exp_max",
        "raw_score",
        "exp_penalty",
        "location_penalty",
        "final_score"
    ]

    cols = [c for c in cols if c in ranked.columns]

    return ranked[cols]

In [ ]:
cv_text = clean_text("software engineer với 2 năm kinh nghiệm phát triển backend đã làm việc với msd python, sql, spark và xây dựng etl pipeline. có kinh nghiệm xây dựng api và xử lý dữ liệu lớn. từng tham gia dự án xây dựng hệ thống data pipeline và xử lý streaming với kafka. quen thuộc với git, docker và quy trình phát triển phần mềm.")
recommend_jobs(cv_text=cv_text,years_experience=3,df_jobs=df,model=model2,job_embeddings=job_embeddings2,candidate_city="hà nội")

,job_id,job_title,company_name,location_city,location_district,experience_required,exp_min,exp_max,raw_score,exp_penalty,location_penalty,final_score
68,69,data engineer,VNG Games,Hà Nội,Đống Đa,2-4 năm,2,4,0.730890,0.000,0.00,0.730890
5,6,data engineer,NAB Innovation Centre,Hồ Chí Minh,Bình Thạnh,2-4 năm,2,4,0.761482,0.000,-0.04,0.721482
70,71,data engineer,Teko,Hồ Chí Minh,Quận 1,1-2 năm,1,2,0.733854,-0.015,-0.04,0.678854
28,29,data engineer,Viettel Digital,Đà Nẵng,Sơn Trà,1-2 năm,1,2,0.729767,-0.015,-0.04,0.674767
29,30,ml engineer,Amanotes,Hà Nội,Đống Đa,1-2 năm,1,2,0.677248,-0.015,0.00,0.662248
55,56,data engineer,Gamota,Hồ Chí Minh,Thủ Đức,4+ năm,4,6,0.750571,-0.050,-0.04,0.660571
80,81,backend developer,Cốc Cốc,Hồ Chí Minh,Bình Thạnh,2-4 năm,2,4,0.695566,0.000,-0.04,0.655566
65,66,data analyst,Appota,Hà Nội,Đống Đa,2-4 năm,2,4,0.653956,0.000,0.00,0.653956
13,14,data engineer,VNPay,Đà Nẵng,Sơn Trà,4+ năm,4,6,0.743391,-0.050,-0.04,0.653391
91,92,backend developer,Sotatek,Đà Nẵng,Ngũ Hành Sơn,1-2 năm,1,2,0.706117,-0.015,-0.04,0.651117
